# Stage 06: merge CH4-to-CO paths into one .nc

Requires a completed `calc-paths-serial`/`calc-paths-parallel` run for
this planet/exp/mdh/tK/grid_resolution first (see
`05_calc_paths_serial.ipynb` or `calc-paths-parallel`) - merges its
per-level JSON Lines into one frozen, shareable `.nc` file.

In [ ]:
import warnings

# What: this notebook is a thin driver.
# Why: numerical/I-O logic lives in quenchpath.stages.stage_06_merge_paths
# (I/O driver) and quenchpath.analysis.path_index (pure pathway-identity/
# ranking logic, reusable outside this stage).
from quenchpath.config import GridResolution, RunConfig
from quenchpath.stages.stage_06_merge_paths import merge_and_save_paths

In [2]:
warnings.filterwarnings("ignore", module="iris")

In [3]:
# What: cfg/grid_resolution here.
# Why: must match a completed calc-paths/calc-paths-parallel run's own
# planet/exp/mdh/tK/grid_resolution exactly (see
# quenchpath.io.paths_dir()).
planet = "hd189733b"
exp = "kin"
mdh = "0"
tK = "1"
cfg = RunConfig(planet=planet, exp=exp, mdh=mdh, tK=tK)
grid_resolution = GridResolution.SAMPLE

In [ ]:
# What: load every saved per-level JSON Lines file, merge into 5
# gridded cubes (temp, pres, path_index, net_rate_inv, max_rate_inv),
# and save as one .nc file with the full pathway identity embedded
# (see quenchpath.analysis.path_index.path_index_map_from_locations()).
# Also returns 2 plain dicts (not cubes) for immediate use below
# without a separate reload: identity_map (the same dict just
# embedded on the path_index cube) and seq_to_index, which isn't
# saved anywhere - it's the {eqn sequence: path_index} mapping used
# to assign path_index in the first place; its length (used in the
# print() below) is the number of distinct pathways found.
# Why embed identity: so a downstream consumer can tell which
# reactions/species a path_index represents without re-running the
# pathway search.
(
    temp_cube,
    pres_cube,
    path_index_cube,
    net_rate_inv_cube,
    max_rate_inv_cube,
    seq_to_index,
    identity_map,
) = merge_and_save_paths(cfg, grid_resolution)
print(f"{temp_cube.data.size} locations merged into {len(seq_to_index)} pathway indices")